In [ ]:
import numpy as np
import datetime
start=datetime.datetime.now()
print(start)

In [ ]:
import os
cwd = os.getcwd()
print(cwd)

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
load = pd.read_csv(r'C:\Users\tmj89\Desktop\Codes\dataset_train_4_0to200.csv')

In [ ]:
load.head

In [ ]:
load.dtypes

import datetime
load['year'] = pd.DatetimeIndex(load['Delivery Date']).year
load['day'] = pd.DatetimeIndex(load['Delivery Date']).day# Extracting Day from "Delivery Date"
load['month'] = pd.DatetimeIndex(load['Delivery Date']).month   # Extracting Month from "Delivery Date"


a = load['Delivery Date']
a= pd.to_datetime(a)

load['dayofweek'] = a.dt.dayofweek
load['dayofweek']

In [ ]:
load = load.set_index(['Delivery Date']) 
load = load.drop(columns=['year','day','Settlement Point Name','Settlement Point Type'], axis=1)  
# Re-arranging Columns
load = load[['dayofweek','month','Delivery Hour','Temperature in F','Load in Houston', 'Settlement Point Price']]
load.head()

In [ ]:
load.dtypes

In [ ]:
# Removing "," from columns "Load in Houston" , "Ercot Load" and "Settlement Point Price"
load['Load in Houston'] = load['Load in Houston'].astype(str).str.replace(r',', '')
#load['Ercot Load'] = load['Ercot Load'].astype(str).str.replace(r',', '')
load['Settlement Point Price'] = load['Settlement Point Price'].astype(str).str.replace(r',', '')
load.head()


In [ ]:
load['Settlement Point Price'] = load['Settlement Point Price'].astype(float)
load['Temperature in F'] = load['Temperature in F'].astype(float)
#load['Load in Houston'] = load['Load in Houston'].astype(float)
#load['Ercot Load'] = load['Ercot Load'].astype(float)
load.dtypes

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(25,15))
plt.hist(load['Settlement Point Price'], bins = (0,5,10,15,20,25,30,35,40,45,50,55,60))
plt.xticks(np.arange(0,61,5))
plt.yticks(np.arange(0,70000,3000))
plt.xlabel("Price")
_ = plt.ylabel("Count")

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, QuantileTransformer, PowerTransformer
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer([("1",OneHotEncoder(sparse=False),make_column_selector(pattern = 'month')),
                             ("2",OneHotEncoder(sparse=False),make_column_selector(pattern = 'dayofweek')),
                             ("3",OneHotEncoder(sparse=False),make_column_selector(pattern = 'Delivery Hour')),
                             #("4",OneHotEncoder(sparse=False),make_column_selector(pattern = 'Delivery Interval')),
                             #("6",OneHotEncoder(sparse=False),make_column_selector(pattern = 'DayStatus')),
                             ("7",StandardScaler(copy=True), make_column_selector(pattern = 'Temperature in F')),
                             ("8",StandardScaler(copy=True), make_column_selector(pattern = 'Load in Houston')),
                             #("9",StandardScaler(copy=True), make_column_selector(pattern = 'Ercot Load')),
                             #("10",StandardScaler(copy=True),make_column_selector(pattern = 'Rise')),
                             ("y",StandardScaler(copy=True), make_column_selector(pattern = 'Settlement Point Price'))])

In [ ]:
import numpy
dataset=load.values
#numpy.around(dataset, decimals=2, out=None)
normed_data = ct.fit_transform(load)
print(normed_data.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# splitting dataset into 'training dataset' and 'validation and test dataset'
Train, Test = train_test_split(normed_data, test_size=0.1, shuffle = False)

print(Train.shape, Test.shape)

In [ ]:
Test[0]

In [ ]:
train_labels = Train[:,(Train.shape[1]-1)]

In [ ]:
test_labels = Test[:,(Test.shape[1]-1)]

In [ ]:
print(train_labels.shape, test_labels.shape)

In [ ]:
train_labels

In [ ]:
test_labels

In [ ]:
Xtrain = Train[:,0:(Train.shape[1]-1)]
Xtrain[0]

In [ ]:
Xtest = Test[:,0:(Test.shape[1]-1)]
Xtest[0]

In [ ]:
from sklearn import svm
start=datetime.datetime.now()
print(start)
clf = svm.SVR()
clf.fit(Xtrain,train_labels)
stop=datetime.datetime.now()
print(stop)

In [ ]:
import datetime
c=datetime.datetime.now()
print(c)
test_predictions = clf.predict(Xtest)
op_list= []
for i in range (0, len(test_predictions)):
  op_list.append((test_predictions[i]))
len(op_list)

In [ ]:
mm= []
for i in range (0, len(test_labels)):
  mm.append((test_labels[i]))
test_labels=pd.DataFrame({'actual':mm})
test_labels['actual']=ct.named_transformers_['y'].inverse_transform(test_labels)
test_labels['actual']

In [ ]:
predictions=pd.DataFrame({'predict':op_list})
predictions['predict'] = ct.named_transformers_['y'].inverse_transform(predictions)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(25,15))
a = plt.axes(aspect='equal')
plt.scatter(test_labels['actual'], predictions)
plt.xlabel('True Value')
plt.ylabel('Predictions')
lims = [0, 200]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims, lims)

In [ ]:
error =predictions['predict']- test_labels['actual']
plt.figure(figsize=(25,15))
plt.hist(error, bins = 50)
plt.xlabel("Prediction Error")
_ = plt.ylabel("Count")

In [ ]:
print(predictions[0:10], test_labels['actual'][0:10], error[0:10])

In [ ]:
i=0
count=0
for i in range(len(error)):
    if ((error[i])>3) or ((error[i]<-3)):
        count=count+1
print(count)

In [ ]:
a = len(test_predictions) - count
a
accuracy = (a/len(test_predictions))*100
print('accuracy:', accuracy)

In [ ]:
from statistics import mean, stdev
print('Mean of error in test data:', mean(error))
print('Standard deviation of error in test data:', stdev(error))

In [ ]:
from statistics import mean, stdev
print('Mean of error in test data:', mean(np.abs(error)))
print('Standard deviation of error in test data:', mean(np.square(error)))

In [ ]:
import datetime
stop=datetime.datetime.now()
print(stop)

In [ ]:
print(stop-start)

In [ ]:
from statistics import mean, stdev
print('Mean of test data:', mean(test_labels['actual']))
print('Standard deviation of test data:', stdev(test_labels['actual']))

In [ ]:
for j in range(1,11):
    i=0
    count=0
    for i in range(len(error)):
        if ((error[i])>j) or ((error[i]<-j)):
            count=count+1
    a = len(predictions) - count
    accuracy = (a/len(predictions))*100
    print("accuracy: with threshold of %i"%j," is %.2f"%accuracy)